# Laboratório 14: Cortando o Ruído com SelectFromModel
**Disciplina:** Extração e Preparação de Dados (IBM8915)  
**Tópico:** Feature Selection (Seleção de Atributos)

Neste laboratório, nós aprenderemos sobre a *Maldição da Dimensionalidade*. Ter colunas demais no dataset (muitas vezes devido a One-Hot Encoding ou explosão de Datetimes) não é sempre bom. Predificadores irrelevantes atuam como *ruído* no modelo.

A missão aqui é usar um **modelo intrínseco (juiz)** para julgar quais colunas valem a pena manter, deletando as inúteis simultaneamente.\

### 0. O Super-Dataset (O Monstro da Dimensionalidade)
Vamos gerar um dataset sintético com 10.000 amostras e **100 colunas (features)**.
Deste montante, apenas 10 colunas são preditores genuínos (Informative). As outras 90 colunas são puro lixo, ruído ou cópias redundantes.\

In [2]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Criando 100 colunas (10 úteis, 90 inúteis)
X, y = make_classification(n_samples=10000, n_features=100, n_informative=10, 
                           n_redundant=40, n_repeated=10, n_classes=2, random_state=42)

# Transformando em um DataFrame Pandas com nomes contínuos de Feature_0 a Feature_99
colunas = [f'Feature_{i}' for i in range(100)]
df = pd.DataFrame(X, columns=colunas)
df['Target'] = y

print(f"Formato Original do Dataset Inteiro: {df.shape}")
df.head()

Formato Original do Dataset Inteiro: (10000, 101)


,Feature_0,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,...,Feature_91,Feature_92,Feature_93,Feature_94,Feature_95,Feature_96,Feature_97,Feature_98,Feature_99,Target
0,0.449176,2.231287,2.784132,-0.986400,-0.488346,-4.603830,0.635236,0.232386,0.635236,0.191417,...,-0.826269,-0.160843,2.200920,-1.790551,3.430373,0.084552,-0.013541,-0.136042,0.836549,1
1,-0.370640,-0.707657,4.614722,1.572847,5.003446,0.579132,-1.549556,-2.207238,-1.549556,-0.442220,...,-1.431287,4.865977,0.099777,-1.818492,0.460964,0.825211,-1.108887,-1.156507,-0.525005,1
2,0.229588,1.170252,1.134208,1.064358,1.309315,-2.330414,-2.266616,0.684564,-2.266616,0.861393,...,-0.274714,1.185990,2.385029,-0.193121,1.264922,2.466412,0.670653,0.599076,-0.328151,0
3,0.748809,2.826923,0.368142,1.208026,3.453290,4.014343,2.564989,0.103267,2.564989,-2.776924,...,-1.315541,2.649309,-3.191705,-0.699229,1.172929,-2.720146,-0.123155,0.989743,1.712217,0
4,-0.769975,-5.321050,-2.220250,-0.983787,0.632546,-3.468896,-3.950360,-0.382240,-3.950360,-0.296600,...,-1.200445,1.409142,4.696065,2.973351,0.697842,3.183438,-1.168327,-0.878328,0.482987,1


### 1. Preparação: Split Seguro
Antes de aplicar qualquer Seleção de Atributos, você DEVE dividir os dados entre Treino e Teste. 
**Atenção:** Rodar Feature Selection na base inteira antes do split é um erro gravíssimo (Data Leakage), pois o algoritmo "espiaria" os dados do futuro para ver o que será útil prever.\

In [3]:
X_dados = df.drop(columns=['Target'])
y_dados = df['Target']

# Aplicando o Split
X_treino, X_teste, y_treino, y_teste = train_test_split(X_dados, y_dados, test_size=0.20, random_state=42)

print(f"X_treino antes da faxina: {X_treino.shape} (10 mil linhas, 100 features completas)!")

X_treino antes da faxina: (8000, 100) (10 mil linhas, 100 features completas)!


### 2. O Juiz: `SelectFromModel`
Nossa missão é instanciar uma árvore de decisão poderosa (`RandomForestClassifier`) e usá-la como argumento dentro da classe de validação contínua de features `SelectFromModel`.\

In [4]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# 1. Instanciamos o modelo Intrínseco que será nosso juiz (A árvore fará o peso e ranking)
modelo_juiz = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Instanciamos a peneira chamando nossa árvore juíza
seletor = SelectFromModel(estimator=modelo_juiz)

### 3. A Faxina Implacável
Agora aplicamos o `.fit_transform()` apenas na base de **TREINO**.
Isso forçará a árvore a estudar o alvo e matar todas as colunas que têm uma "nota" (importância) abaixo da média geral.\

In [5]:
# 3. Aplicamos no Treino (estuda o alvo e corta as features irrelevantes na mesma linha de código!)
X_treino_limpo = seletor.fit_transform(X_treino, y_treino)

print(f"Atenção à Redução Dimensional!")
print(f"X_treino Antes:  {X_treino.shape}")
print(f"X_treino Depois: {X_treino_limpo.shape}")

Atenção à Redução Dimensional!
X_treino Antes:  (8000, 100)
X_treino Depois: (8000, 32)


### 4. Extraindo as Sobreviventes
O processo nos retornou uma matriz numpy. Se quisermos transformá-la num DataFrame limpo novamente, precisamos descobrir o "nome" das colunas que sobreviveram usando o atributo `.get_support()`, que retorna uma máscara Booleana (`True` = sobreviveu, `False` = deletada).\

In [6]:
# Máscara binária de quem sobreviveu
sobreviventes_mask = seletor.get_support()

# Buscando os nomes originais da matriz de treino baseada na máscara
nomes_sobreviventes = X_treino.columns[sobreviventes_mask]

# Visualizando as elites (Notem como ele reduziu 100 features para próximo de 20 a 30!)
print("As Features de Elite que sobreviveram ao juiz:")
print(nomes_sobreviventes.to_list())

# Reconstruindo o DataFrame de Treino limpo
df_treino_limpo = pd.DataFrame(X_treino_limpo, columns=nomes_sobreviventes)
df_treino_limpo.head(3)

As Features de Elite que sobreviveram ao juiz:
['Feature_1', 'Feature_4', 'Feature_5', 'Feature_17', 'Feature_19', 'Feature_22', 'Feature_27', 'Feature_30', 'Feature_33', 'Feature_34', 'Feature_38', 'Feature_40', 'Feature_41', 'Feature_44', 'Feature_48', 'Feature_52', 'Feature_55', 'Feature_57', 'Feature_59', 'Feature_60', 'Feature_74', 'Feature_75', 'Feature_76', 'Feature_77', 'Feature_78', 'Feature_82', 'Feature_83', 'Feature_87', 'Feature_89', 'Feature_92', 'Feature_94', 'Feature_96']


,Feature_1,Feature_4,Feature_5,Feature_17,Feature_19,Feature_22,Feature_27,Feature_30,Feature_33,Feature_34,...,Feature_76,Feature_77,Feature_78,Feature_82,Feature_83,Feature_87,Feature_89,Feature_92,Feature_94,Feature_96
0,-2.944984,-8.787233,-3.684868,-0.734197,3.124745,-1.382490,-0.698429,6.325159,1.440882,-0.771367,...,-3.784798,-1.224936,2.991221,0.003082,-0.892764,-0.557920,0.479939,-8.854934,1.372675,3.591789
1,3.806722,0.203630,9.844493,-3.318078,4.433635,2.230758,-4.913988,2.527173,-1.531468,-2.408654,...,3.383822,1.377805,3.633046,8.138008,4.239774,0.602233,-1.592467,1.154961,-5.575953,-2.168810
2,3.265682,4.200090,3.444167,0.117246,0.046271,4.124728,-2.064852,0.185432,-1.200799,-3.472124,...,3.919771,1.696024,1.313901,2.136383,-0.750948,-0.858123,-0.660708,3.687819,-5.413100,-0.435166


**Missão Cumprida!** Vocês eliminaram o excesso estatístico (Ruído) do modelo de vocês, melhorando a eficácia e drásticamente a velocidade de treinamento e processamento futuramente na esteira no Airflow!  
Salvem o notebook e vamos ao **Lab 15** conhecer a abordagem mais cirúrgica (porém pesada): o **RFE**.\

---
## 🚀 Desafio Prático: O Juiz Implacável no Mundo Real

Lembra do nosso dataset de Cartão de Crédito ([Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud))? Ele tem 30 colunas (V1 a V28, Time e Amount).
É muito provável que várias dessas colunas derivadas de PCA não ajudem em nada na detecção de fraudes, apenas consumindo memória.

**Sua Missão:**
1. Carregue o `creditcard.csv` e isole o `X` e `y`.
2. Faça o `train_test_split`.
3. Instancie um `RandomForestClassifier` e use-o dentro de um `SelectFromModel`.
4. Transforme seu `X_treino`. O Random Forest cortou quantas features das 30 originais? Use `.get_support()` para descobrir quais colunas essenciais ele manteve!\

In [ ]:
# IMPLEMENTE SEU CÓDIGO AQUI

# 1. Carregue os dados
df_kaggle = pd.read_csv('creditcard.csv')
print(f"Shape original: {df_kaggle.shape}")

# 2. Separar X e y
X_kag = df_kaggle.drop(columns=['Class'])
y_kag = df_kaggle['Class']

# train_test_split
X_kag_treino, X_kag_teste, y_kag_treino, y_kag_teste = train_test_split(
    X_kag, y_kag, test_size=0.20, random_state=42
)

# 3. Instancie e aplique o SelectFromModel com RandomForestClassifier
seletor_kaggle = SelectFromModel(estimator=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
X_kag_treino_limpo = seletor_kaggle.fit_transform(X_kag_treino, y_kag_treino)

# 4. Descubra quais features sobreviveram
sobreviventes = X_kag_treino.columns[seletor_kaggle.get_support()]
print(f"\nFeatures originais: {X_kag_treino.shape[1]}")
print(f"Features selecionadas: {len(sobreviventes)}")
print(f"\nFeatures que sobreviveram ao corte:")
print(sobreviventes.to_list())

---
## 🧠 Questão Discursiva

**Pergunta:** Nós acabamos de usar o `SelectFromModel` (baseado em Árvore). Existe uma outra família de seleção chamada de *Filtros Estatísticos Básicos* (ex: teste de correlação de Pearson de cada coluna com o alvo). 

Qual a vantagem principal do método que usamos no laboratório de hoje em comparação a um simples filtro de correlação matemática isolada?\

**SUA RESPOSTA AQUI:**
*(Dê um duplo-clique para editar esta célula e escreva sua resposta).*\